# MagNet Standalone Notebook

This notebook bundles the MagNet project `main.py` and its dependencies.
It is designed to run in environments like Google Colab without requiring file uploads.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Install dependencies
#!pip install -r requirements.txt
import os
import sys
# Add current directory to path just in case
sys.path.append(os.getcwd())

### Configuration
Creating `config/config.yaml`...

In [4]:
import os
os.makedirs('config', exist_ok=True)
config_content = """# Configuration parameters for MagNet ML Project

# Data settings
data:
  raw_path: "/content/config/raw"
  processed_path: "/content/config/processed"
  batch_size: 32
  test_split: 0.2

# Model settings
models:
  scaler:
    hidden_dim: 64
    layers: 3
    features:
      inputs:
        B: standard
        H: standard
        Frequency: standard
        Temperature: standard
        Hdc: standard
      targets:
        Loss: standard
  sequence:
    hidden_dim: 128
    num_layers: 2
    features:
      inputs:
        B: standard
        Frequency: standard
        Temperature: standard
        Hdc: standard
      targets:
        Loss: standard
  seq2seq:
    encoder_dim: 128
    decoder_dim: 128
    features:
      inputs:
        B: standard
        H: standard
      targets:
        Loss: standard
  cnn:
    kernel_size: 3
    num_channels: 64
    num_layers: 3
    features:
      inputs:
        B: minmax
        Frequency: standard
        Temperature: standard
        Hdc: standard
      targets:
        Loss: standard
  transformer:
    d_model: 64
    nhead: 4
    num_layers: 2
    dim_feedforward: 128
    dropout: 0.1
    features:
      inputs:
        B: standard
        Frequency: standard
        Temperature: standard
        Hdc: standard
      targets:
        Loss: standard

# Training settings
training:
  epochs: 100
  learning_rate: 0.001
  save_dir: "checkpoints"
"""
with open('config/config.yaml', 'w') as f:
    f.write(config_content)

### Module: `src\data\loader.py`
Content from local file: `src\data\loader.py`

In [5]:
"""
Data Loading Module.

This module is responsible for loading experimental data from MATLAB (.mat) files.
It handles both legacy (SciPy) and v7.3 (HDF5/h5py) formats.

Functions:
- load_experiment(file_path, experiment_id): Loads data for a single experiment.
"""

import numpy as np
import scipy.io
import h5py
import os

def load_experiment(file_path, experiment_id):
    """
    Loads data for a specific experiment ID from a .mat file.

    Args:
        file_path (str): Path to the .mat file.
        experiment_id (int): 1-based experiment index (as in MATLAB).

    Returns:
        dict: Dictionary containing experimental data (Voltage, Current, Metadata).
    """
    idx = experiment_id - 1  # Convert to 0-based index

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        # Try loading with scipy (for v7 and earlier)
        mat = scipy.io.loadmat(file_path, squeeze_me=True, struct_as_record=False)
        Data = mat['Data']

        def get_val(obj, idx):
            if np.ndim(obj) > 0:
                return obj[idx]
            return obj

        data_dict = {
            'voltage': Data.Voltage[idx, :],
            'current': Data.Current[idx, :],
            'primary_turns': Data.Primary_Turns,
            'secondary_turns': Data.Secondary_Turns,
            'effective_area': Data.Effective_Area,
            'effective_length': Data.Effective_Length,
            'sampling_time': get_val(Data.Sampling_Time, idx),
            'hdc': get_val(Data.Hdc_command, idx),
            'temperature': get_val(Data.Temperature_command, idx),
            'duty_p': get_val(Data.DutyP_command, idx),
            'frequency': get_val(Data.Frequency_command, idx),
            'flux_cmd': get_val(Data.Flux_command, idx)
        }
        return data_dict

    except NotImplementedError:
        # Fallback to h5py for v7.3 files
        with h5py.File(file_path, 'r') as f:
            Data = f['Data']

            # Helper to access h5py data
            # Arrays are transposed in h5py relative to MATLAB
            # MATLAB: Voltage(ExperimentID, :) -> h5py: Voltage[Samples, ExperimentID]
            # We want all samples for the specific experiment column

            v_sec = Data['Voltage'][:, idx]
            i_prim = Data['Current'][:, idx]

            def get_scalar(key):
                val = np.array(Data[key])
                if val.size == 1:
                    return val.item()
                return val

            def get_indexed(key, idx):
                val = np.array(Data[key]).flatten()
                if len(val) > idx:
                    return val[idx]
                return val.item()

            data_dict = {
                'voltage': v_sec,
                'current': i_prim,
                'primary_turns': get_scalar('Primary_Turns'),
                'secondary_turns': get_scalar('Secondary_Turns'),
                'effective_area': get_scalar('Effective_Area'),
                'effective_length': get_scalar('Effective_Length'),
                'sampling_time': get_indexed('Sampling_Time', idx),
                'hdc': get_indexed('Hdc_command', idx),
                'temperature': get_indexed('Temperature_command', idx),
                'duty_p': get_indexed('DutyP_command', idx),
                'frequency': get_indexed('Frequency_command', idx),
                'flux_cmd': get_indexed('Flux_command', idx)
            }
            return data_dict

def get_dataset_info(file_path):
    """
    Retrieves metadata about the dataset without loading full data.

    Returns:
        num_experiments (int)
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        # Try scipy
        mat = scipy.io.loadmat(file_path, squeeze_me=True, struct_as_record=False)
        # shape might be (Experiments, samples) or transposed.
        # Based on prev code: Data.Voltage was accessed as [idx, :] -> (N_exp, N_samples)
        return mat['Data'].Voltage.shape[0]
    except NotImplementedError:
        with h5py.File(file_path, 'r') as f:
            # h5py: Voltage shape (Samples, Experiments)
            return f['Data']['Voltage'].shape[1]

def load_full_dataset(file_path):
    """
    Loads the entire dataset into memory.
    Optimized for bulk loading (reads full arrays).
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        # SciPy
        mat = scipy.io.loadmat(file_path, squeeze_me=True, struct_as_record=False)
        Data = mat['Data']

        return {
            'voltage': Data.Voltage,   # (N_exp, N_samples)
            'current': Data.Current,
            'freq': Data.Frequency_command,
            'temp': Data.Temperature_command,
            'hdc': Data.Hdc_command,
            'duty': Data.DutyP_command,
            # Add other fields as needed
            'meta': {
               'N_prim': Data.Primary_Turns,
               'N_sec': Data.Secondary_Turns,
               'Ae': Data.Effective_Area,
               'Le': Data.Effective_Length,
               'dt': Data.Sampling_Time
            }
        }
    except NotImplementedError:
        # h5py
        with h5py.File(file_path, 'r') as f:
            Data = f['Data']

            # Read all at once. Transpose to (N_exp, N_samples)
            # h5py stores as (samples, experiments).
            # T operation in numpy is cheap (view), but reading might be slow if chunked.
            # reading as [:] reads into memory.

            print("Reading full arrays from HDF5... this may take a moment.")
            v_all = np.array(Data['Voltage']).T
            i_all = np.array(Data['Current']).T

            # 1D arrays
            def get_arr(key):
                return np.array(Data[key]).flatten()

            data_dict = {
                'voltage': v_all,
                'current': i_all,
                'freq': get_arr('Frequency_command'),
                'temp': get_arr('Temperature_command'),
                'hdc': get_arr('Hdc_command'),
                'duty': get_arr('DutyP_command'),
                'meta': {
                   'N_prim': np.array(Data['Primary_Turns']).item(),
                   'N_sec': np.array(Data['Secondary_Turns']).item(),
                   'Ae': np.array(Data['Effective_Area']).item(),
                   'Le': np.array(Data['Effective_Length']).item(),
                   # Sampling time might be an array or scalar?
                   # Assuming scalar for uniform sampling usually, or array.
                   'dt': np.array(Data['Sampling_Time']).flatten()
                }
            }
            return data_dict


### Module: `src\data\preprocessing.py`
Content from local file: `src\data\preprocessing.py`

In [6]:
"""
Data Preprocessing Module.

Generic processing for Time-Series FNN.
"""
import numpy as np
from scipy.integrate import cumulative_trapezoid, trapezoid

def normalize_data(data, method='standard', stats=None):
    """
    Normalizes data using Global Scaling (Statistics computed over ALL data).

    Args:
        data (np.array): Input data.
        method (str): 'standard' (Z-score), 'minmax', or 'none'.
        stats (dict, optional): Pre-computed stats.

    Returns:
        norm_data (np.array): Normalized data.
        stats (dict): Statistics used.
    """
    # Force float32
    data = data.astype(np.float32)

    if method == 'none':
        return data, {}

    if method == 'standard':
        if stats:
            mean = stats['mean']
            std = stats['std']
        else:
            mean = np.mean(data, axis=None)
            std = np.std(data, axis=None)
            if std == 0: std = 1.0
            stats = {'mean': mean, 'std': std}

        norm_data = (data - mean) / std
        return norm_data, stats

    elif method == 'minmax':
        if stats:
            min_val = stats['min']
            max_val = stats['max']
        else:
            min_val = np.min(data, axis=None)
            max_val = np.max(data, axis=None)
            stats = {'min': min_val, 'max': max_val}

        denom = max_val - min_val
        if denom == 0: denom = 1.0

        norm_data = (data - min_val) / denom
        return norm_data, stats

    else:
        raise ValueError(f"Unknown method: {method}")


def prepare_dataset(features, targets, test_ratio=0.2, norm_config=None):
    """
    Generic function to split and normalize ANY set of features and targets.

    Args:
        features (dict): Dictionary mapping FeatureName to data array.
        targets (dict): Dictionary mapping TargetName to data array.
        test_ratio (float): Fraction of data to use for testing.
        norm_config (dict): Dictionary mapping feature/target names to methods.
                            e.g. {'B': 'standard', 'Loss': 'log10'}.
                            Defaults to 'standard' for features and 'none' for targets if not specified.

    Returns:
        data_split (dict): Contains 'train' and 'test' dictionaries.

                           - `data_split['train']['inputs']['FeatureName']`
                           - `data_split['train']['targets']['TargetName']`

        stats (dict): The statistics used for normalization for each feature/target.
    """
    if norm_config is None:
        norm_config = {}

    # 1. Determine Split Index
    # Assume all arrays have same length N in dimension 0.
    any_key = next(iter(features))
    N = len(features[any_key])
    split_idx = int(N * (1 - test_ratio))

    split_data = {
        'train': {'inputs': {}, 'targets': {}},
        'test': {'inputs': {}, 'targets': {}}
    }
    stats_out = {}

    print(f"Splitting Dataset: {N} Samples -> {split_idx} Train, {N - split_idx} Test")

    # 2. Process Inputs (Features)
    print("Processing Features...")
    for name, data in features.items():
        # Get method (Default to 'standard' for inputs)
        method = norm_config.get(name, 'standard')

        # Split
        train_part = data[:split_idx]
        test_part = data[split_idx:]

        # Normalize Train
        print(f"  - Normalizing '{name}' with {method}...")
        train_norm, stat = normalize_data(train_part, method=method)
        stats_out[name] = stat

        # Normalize Test (using Train stats)
        test_norm, _ = normalize_data(test_part, method=method, stats=stat)

        split_data['train']['inputs'][name] = train_norm
        split_data['test']['inputs'][name] = test_norm

    # 3. Process Targets
    print("Processing Targets...")
    for name, data in targets.items():
        # Get method (Default to 'none' for targets unless specified)
        # Often we don't normalize targets for regression unless specifically asked (like Log Loss)
        method = norm_config.get(name, 'none')

        # Split
        train_part = data[:split_idx]
        test_part = data[split_idx:]

        # Normalize Train
        if method != 'none':
             print(f"  - Normalizing Target '{name}' with {method}...")

        train_norm, stat = normalize_data(train_part, method=method)
        if method != 'none':
            stats_out[name] = stat

        # Normalize Test
        test_norm, _ = normalize_data(test_part, method=method, stats=stat)

        split_data['train']['targets'][name] = train_norm
        split_data['test']['targets'][name] = test_norm

    return split_data, stats_out


# --- PHYSICS FUNCTIONS (Vectorized) ---
# Kept as helpers, but they are just ONE way to generate features.

def calculate_flux_density(voltage, sampling_time, secondary_turns, effective_area):
    """ Calculates B (Flux Density). """
    v_mean = np.mean(voltage, axis=1, keepdims=True)
    v_clean = voltage - v_mean
    flux = cumulative_trapezoid(v_clean, axis=-1, initial=0)

    if np.ndim(sampling_time) == 0:
        flux = flux * sampling_time
    else:
        st = np.array(sampling_time).reshape(-1, 1)
        flux = flux * st

    b = flux / (secondary_turns * effective_area)
    b_mean = np.mean(b, axis=1, keepdims=True)
    b = b - b_mean

    return b.astype(np.float32)

def calculate_magnetizing_force(current, primary_turns, effective_length):
    """ Calculates H (Magnetizing Force). """
    h = (primary_turns * current) / effective_length
    return h.astype(np.float32)

def calculate_volumetric_loss(b_field, h_field, frequency):
    """ Calculates Volumetric Power Loss (Target). """
    energy_density = trapezoid(y=h_field, x=b_field, axis=-1)
    energy_density = np.abs(energy_density)

    if np.ndim(frequency) > 0:
        freq = np.array(frequency).squeeze()
    else:
        freq = frequency

    pv = energy_density * freq
    return pv.astype(np.float32)


# --- WRAPPER FOR FULL MagNet PIPELINE ---

def load_config(config_path):
    import yaml
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

def process_magnet_dataset(voltage, current, frequency, mag_props, dt, model_type='scaler', config_path=None, extra_features=None):
    """
    Orchestrates the physics calculation -> Feature Assembly -> Normalization.

    Args:
        voltage, current: Raw arrays
        frequency: Raw frequency array
        mag_props, dt: Constants
        model_type (str): Key in config['models'] (e.g. 'scaler', 'cnn')
        config_path (str, optional): Path to config.yaml.
        extra_features (dict, optional): Additional raw features (e.g. {'Temperature': ..., 'Hdc': ...})
    """
    norm_config = {}

    # 1. Load Config if path provided
    if config_path:
        full_config = load_config(config_path)
        if 'models' in full_config and model_type in full_config['models']:
            model_conf = full_config['models'][model_type]
            if 'features' in model_conf:
                # Merge inputs and targets into one simple config dict for prepare_dataset
                # prepare_dataset expects {'B': 'method', 'Loss': 'method'}
                feats = model_conf['features'].get('inputs', {})
                targs = model_conf['features'].get('targets', {})
                norm_config = {**feats, **targs}
                print(f"Loaded config for '{model_type}': {norm_config}")
            else:
                print(f"WARNING: No 'features' section found for model '{model_type}'. Using defaults.")
        else:
             print(f"WARNING: Model '{model_type}' not found in config. Using defaults.")

    # Fallback default if empty
    if not norm_config:
        # Added defaults for Temp/Hdc just in case
        norm_config = {'B': 'standard', 'H': 'standard', 'Loss': 'standard', 'Frequency': 'standard',
                       'Temperature': 'standard', 'Hdc': 'standard', 'Duty': 'standard'}

    print("--- 1. Physics Calculations ---")
    B_raw = calculate_flux_density(voltage, dt, mag_props['N_sec'], mag_props['Ae'])
    H_raw = calculate_magnetizing_force(current, mag_props['N_prim'], mag_props['Le'])
    Loss_raw = calculate_volumetric_loss(B_raw, H_raw, frequency)

    # Pack Potential Features
    all_features = {
        'B': B_raw,
        'H': H_raw,
        'Frequency': frequency.reshape(-1, 1) if np.ndim(frequency) > 0 else np.full((len(B_raw), 1), frequency)
    }

    if extra_features:
        # Reshape scalers if needed?
        # Assuming extra_features are (N_exp,) or (N_exp, 1) or matching B_raw length
        # Let's ensure they are at least 2D (N, 1) if they are scalars per experiment
        for k, v in extra_features.items():
            if np.ndim(v) == 1:
                all_features[k] = v.reshape(-1, 1)
            else:
                all_features[k] = v

    all_targets = {
        'Loss': Loss_raw
    }

    # 2. Filter Features/Targets based on Config
    # We only include keys that are in the norm_config
    selected_features = {k: v for k, v in all_features.items() if k in norm_config}
    selected_targets = {k: v for k, v in all_targets.items() if k in norm_config}

    if not selected_features:
        print("WARNING: No features selected! Checking config...")
        # Fallback to keeping all if config was malformed to avoid returning nothing
        selected_features = all_features

    print(f"Selected Features: {list(selected_features.keys())}")
    print(f"Selected Targets: {list(selected_targets.keys())}")

    print("--- 2. Generic Normalization & Split ---")
    data_split, stats = prepare_dataset(selected_features, selected_targets, test_ratio=0.2, norm_config=norm_config)

    return data_split, stats


### Module: `src\data\dataset.py`
Content from local file: `src\data\dataset.py`

In [7]:
"""
MagNet Dataset Module.

This module provides a PyTorch Dataset wrapper for the MagNet data.
It handles:
1. Loading the full dataset into memory.
2. Computing derived quantities (B, H, Power Loss).
3. Normalizing features.
4. Serving data based on the requested mode ('scaler', 'sequence', 'seq2seq').

Classes:
- MagNetDataset(Dataset)
"""

import torch
from torch.utils.data import Dataset
import numpy as np


class MagNetDataset(Dataset):
    def __init__(self, file_path, mode='scaler', partition='train', transform=None, config_path=None):
        """
        Args:
            file_path (str): Path to .mat file.
            mode (str): 'scaler', 'sequence', 'seq2seq'.
            partition (str): 'train' or 'test'.
            transform (callable, optional): Optional transform to be applied.
            config_path (str): Path to config.yaml (optional).
        """
        self.mode = mode
        self.partition = partition
        self.transform = transform

        # Load raw data
        print(f"Loading dataset from {file_path} for partition '{partition}'...")
        raw_data = load_full_dataset(file_path)

        # Extract Standard Args
        voltage = raw_data['voltage'].astype(np.float32)
        current = raw_data['current'].astype(np.float32)
        freq = raw_data['freq'].astype(np.float32)

        props = {
            'N_prim': raw_data['meta']['N_prim'],
            'N_sec': raw_data['meta']['N_sec'],
            'Ae': raw_data['meta']['Ae'],
            'Le': raw_data['meta']['Le']
        }
        dt = raw_data['meta']['dt']
        if isinstance(dt, (list, np.ndarray)) and len(dt) > 1:
            dt = dt[0]

        # Extra Features
        extra = {
            'Temperature': raw_data['temp'].astype(np.float32),
            'Hdc': raw_data['hdc'].astype(np.float32),
            'Duty': raw_data['duty'].astype(np.float32)
        }

        # Use Central Preprocessing
        print("Running centralized preprocessing...")

        # Determine config path if not provided
        if not config_path:
             # Try to find it relative to this file
             import os
             #current_dir = os.path.dirname(os.path.abspath(__file__))
             # Scripts/src/data -> ... -> Scripts/config/config.yaml
             config_path = os.path.abspath( '/content/config/config.yaml')

        if not os.path.exists(config_path):
            print(f"WARNING: Config not found at {config_path}. Using defaults.")
            config_path = None

        data_split, self.stats = process_magnet_dataset(
            voltage, current, freq, props, dt,
            model_type=mode,
            config_path=config_path,
            extra_features=extra
        )

        # Select Partition
        if partition not in data_split:
            raise ValueError(f"Partition '{partition}' not found in split data.")

        self.inputs = data_split[partition]['inputs']
        self.targets = data_split[partition]['targets']

        # Validation checks
        if not self.targets:
             print("WARNING: No targets found in processed data.")

        if not self.inputs:
             print("WARNING: No inputs found in processed data.")

        # Store length
        # Assuming all input arrays are same length
        any_key = next(iter(self.inputs)) if self.inputs else next(iter(self.targets))
        self.length = len(self.inputs[any_key]) if self.inputs else len(self.targets[any_key])

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Retrieve normalized features from dictionary
        # NOTE: Keys depend on config.yaml!
        # We must align code here with what we expect to be in the config for each mode.
        # Fallback logic is needed if config is dynamic.

        # Helper to get datum safely
        def get(dic, key):
            if key in dic:
                val = dic[key][idx]
                return torch.tensor(val, dtype=torch.float32)
            else:
                 # Be noisy if missing expected feature
                 raise KeyError(f"Feature '{key}' missing from dataset. Check config.yaml or preprocessing.")

        # Helper to get sequence or scalar
        def get_tens(dic, key, unsqueeze=False):
            t = get(dic, key)
            if unsqueeze:
                return t.unsqueeze(-1)
            return t

        if self.mode == 'scaler':
            # Config MUST include Frequency, Temperature, Hdc in inputs
            # Config MUST include Loss in targets

            # Input: Freq, Temp, Hdc
            # We assume these are 1D arrays (scalers) in the input dict
            f = get(self.inputs, 'Frequency')
            t = get(self.inputs, 'Temperature')
            h = get(self.inputs, 'Hdc')

            x = torch.stack([f.squeeze(), t.squeeze(), h.squeeze()])
            # Note: normalized scalars might come as (1,) or scalar. Squeeze ensures (3,)

            y = get_tens(self.targets, 'Loss', unsqueeze=False) # (1,)
            return x, y

        elif self.mode in ['sequence', 'cnn', 'transformer']:
            # Input: B (or H)
            # Target: Loss
            b = get_tens(self.inputs, 'B', unsqueeze=True) # (Seq, 1)

            # Scalars: Freq, Temp, Hdc
            f = get(self.inputs, 'Frequency')
            t = get(self.inputs, 'Temperature')
            h = get(self.inputs, 'Hdc')
            scalars = torch.stack([f.squeeze(), t.squeeze(), h.squeeze()])

            y = get_tens(self.targets, 'Loss', unsqueeze=False)
            return b, scalars, y

        elif self.mode == 'seq2seq':
            # Input: B
            # Target: H
            b = get_tens(self.inputs, 'B', unsqueeze=True)
            h = get_tens(self.inputs, 'H', unsqueeze=True) # Assuming H is in inputs or targets?
            # Usually H is 'target' for B->H prediction? Or Input?
            # If we want to predict H, it should be in targets.
            # But process_magnet_dataset puts things in 'inputs' or 'targets' based on config.
            # Check where H is.

            if 'H' in self.targets:
                target = get_tens(self.targets, 'H', unsqueeze=True)
            elif 'H' in self.inputs:
                 target = get_tens(self.inputs, 'H', unsqueeze=True)
            else:
                raise KeyError("H field not found in inputs or targets for seq2seq")

            return b, target

        else:
            raise ValueError(f"Unknown mode: {self.mode}")



### Module: `src\models\scaler_model.py`
Content from local file: `src\models\scaler_model.py`

In [8]:
"""
Scaler-to-Scaler Model.

This module implements a standard Multi-Layer Perceptron (MLP) for predicting
scalar outputs (e.g., Power Loss) from scalar inputs (Frequency, Temperature, Hdc).
"""

import torch
import torch.nn as nn

class ScalerNetwork(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, output_dim=1, num_layers=3):
        """
        Args:
            input_dim (int): Number of input features (default 3: Freq, Temp, Hdc).
            hidden_dim (int): Number of neurons in hidden layers.
            output_dim (int): Number of output features (default 1: Power Loss).
            num_layers (int): Number of hidden layers.
        """
        super(ScalerNetwork, self).__init__()

        layers = []

        # Input Layer
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.BatchNorm1d(hidden_dim))

        # Hidden Layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))

        # Output Layer
        layers.append(nn.Linear(hidden_dim, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

### Module: `src\models\sequence_model.py`
Content from local file: `src\models\sequence_model.py`

In [9]:

import torch
import torch.nn as nn

class SequenceToScalerNetwork(nn.Module):
    """
    Bristol's LSTM Seq2One Architecture.
    """
    def __init__(self,
                 input_dim=1, # Ignored, hardcoded to 3 (B, Freq, Temp)
                 hidden_dim=30,
                 output_dim=1, # Power Loss
                 num_layers=3):
        super(SequenceToScalerNetwork, self).__init__()

        self.hidden_size = hidden_dim

        # Bristol uses input_size=1 but constructs a tensor of size 3 (B, F, T) inside, NOT quite.
        # Bristol's code actually expected input of size 3 (B, Freq, Temp) from the start?
        # "inputs = torch.zeros(64, waveStep, 3)" -> Yes.
        # But their LSTM init says input_size=1?
        # "self.lstm = nn.LSTM(input_size, ..." -> If they used input_size=1, they only fed B?
        # Let's check their forward: "out, _ = self.lstm(in_b)" -> in_b is x[:,:,0:1].
        # SO LSTM ONLY SEES B-Field!

        # LSTM layer (Processing B-Field Only)
        self.lstm = nn.LSTM(1, # Fixed to 1 for B-field
                            hidden_dim,
                            num_layers=num_layers,
                            batch_first=True)

        # Fully connected layer
        # Input to FC is: LSTM_Out + Freq + Temp = hidden_dim + 2
        self.fc1 = nn.Linear(hidden_dim+2, 128)
        self.fc2 = nn.Linear(128, 196)
        self.fc3 = nn.Linear(196, 128)
        self.fc4 = nn.Linear(128, 96)
        self.fc5 = nn.Linear(96, 32)
        self.fc6 = nn.Linear(32, 32)
        self.fc7 = nn.Linear(32, 16)
        self.fc8 = nn.Linear(16, output_dim)

        # Activation function
        self.elu = nn.ELU()


    def forward(self, b_seq, scalars):
        """
        Unified Interface Wrapper.

        Args:
            b_seq (Tensor): (Batch, Seq, 1)
            scalars (Tensor): (Batch, 3) -> Freq, Temp, Hdc
        """

        # Unpack scalars
        Freq = scalars[:, 0].unsqueeze(1) # (bs, 1)
        Temp = scalars[:, 1].unsqueeze(1) # (bs, 1)

        # Bristol's logic:
        # 1. Feed only B-field (b_seq) into LSTM
        # out: (Batch, Seq, Hidden)
        out, _ = self.lstm(b_seq)

        # 2. Take last output
        out = out[:, -1, :]  # (Batch, Hidden)

        # 3. Concatenate Freq and Temp to the features
        out = torch.cat([out, Freq, Temp], dim=1) # (Batch, Hidden + 2)

        # 4. Deep MLP
        out = self.fc1(out)
        out = self.elu(self.fc2(out))
        out = self.elu(self.fc3(out))
        out = self.elu(self.fc4(out))
        out = self.elu(self.fc5(out))
        out = self.elu(self.fc6(out))
        out = self.fc7(out)
        out = self.fc8(out)

        return out

### Module: `src\models\seq2seq_model.py`
Content from local file: `src\models\seq2seq_model.py`

In [10]:
"""
Sequence-to-Sequence Model.

This module implements Encoder-Decoder architectures for mapping input waveforms
(Excitation, e.g., H) to output waveforms (Response, e.g., B).
"""

import torch
import torch.nn as nn

class Seq2SeqNetwork(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, output_dim=1, num_layers=2):
        """
        Args:
            input_dim (int): Input feature size.
            hidden_dim (int): Hidden size.
            output_dim (int): Output feature size.
            num_layers (int): Depth of LSTM.
        """
        super(Seq2SeqNetwork, self).__init__()

        # Encoder
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1
        )

        # Decoder
        # Input to decoder is previous output (or ground truth in training).
        # We model this as mapping Hidden State -> Sequence.

        # Simple Approach: Use LSTM to map (Batch, Seq, Hidden) -> (Batch, Seq, Out).
        # But standard Seq2Seq uses an Decoder LSTM.

        # Here we implement a simple LSTM-based mapping (Many-to-Many).
        # Since input and output length are same for hysteresis loops.
        # This acts like a Bi-Directional LSTM or just a mapped LSTM.

        self.decoder = nn.LSTM(
            input_size=hidden_dim, # We might feed encoder outputs or similar
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1
        )

        self.head = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (Batch, Seq, Input)

        # Encoder
        # We want to map Sequence -> Sequence.
        # If lengths are same and it's 1:1 mapping (like filtering),
        # a single LSTM (Encoder) + Linear Head per step is sufficient.

        # output: (Batch, Seq, Hidden)
        enc_out, _ = self.encoder(x)

        # Decode/Map
        # dec_out, _ = self.decoder(enc_out) # Optional: Deepen model

        # Project to output
        # out: (Batch, Seq, Output)
        out = self.head(enc_out)

        return out

### Module: `src\models\cnn_model.py`
Content from local file: `src\models\cnn_model.py`

In [11]:

import torch
import torch.nn as nn
import numpy as np

# ==========================================
# Paderborn's TCN Architecture Components
# ==========================================

class Biased_Elu(nn.Module):
    def __init__(self):
        super().__init__()
        self.elu = nn.ELU()

    def forward(self, x):
        return self.elu(x) + 1

class SinusAct(nn.Module):
    def forward(self, x):
        return torch.sin(x)

class GeneralizedCosinusUnit(nn.Module):
    def forward(self, x):
        return torch.cos(x) * x

ACTIVATION_FUNCS = {
    "sigmoid": nn.Sigmoid,
    "tanh": nn.Tanh,
    "relu": nn.ReLU,
    "biased_elu": Biased_Elu,
    "sinus": SinusAct,
    "gcu": GeneralizedCosinusUnit,
}

class TemporalBlock(nn.Module):
    def __init__(
        self,
        n_inputs,
        n_outputs,
        kernel_size,
        stride,
        dilation,
        residual=True,
        double_layered=True,
        dropout=0.0,
        act_func=None,
    ):
        super(TemporalBlock, self).__init__()
        padding = ((kernel_size - 1) // 2) * dilation

        self.conv1 = nn.utils.weight_norm(
            nn.Conv1d(
                n_inputs,
                n_outputs,
                kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                padding_mode="circular",
            )
        )
        self.relu1 = ACTIVATION_FUNCS.get(act_func, nn.Identity)()
        self.dropout1 = nn.Dropout1d(dropout)
        if double_layered:
            self.relu2 = nn.Identity()
            self.conv2 = nn.utils.weight_norm(
                nn.Conv1d(
                    n_inputs,
                    n_outputs,
                    kernel_size,
                    stride=stride,
                    padding=padding,
                    dilation=dilation,
                    padding_mode="circular",
                )
            )
            self.dropout2 = nn.Dropout1d(dropout)
            self.net = nn.Sequential(
                self.conv1,
                self.relu1,
                self.dropout1,
                self.conv2,
                self.relu2,
                self.dropout2,
            )
        else:
            self.net = nn.Sequential(self.conv1, self.relu1, self.dropout1)
        self.relu = nn.ReLU()
        self.residual = residual
        if residual:
            self.downsample = (
                nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
            )
        else:
            self.downsample = None
        self.double_layered = double_layered
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        if self.double_layered:
            self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        if self.residual:
            res = x if self.downsample is None else self.downsample(x)
            y = torch.clip(
                out + res, -10, 10
            )
            y = self.relu(y)
        else:
            y = out
        return y


class TCNWithScalarsAsBias(nn.Module):
    def __init__(
        self,
        num_input_scalars,
        num_input_ts=1,
        tcn_layer_cfg=None,
        scalar_layer_cfg=None,
    ):
        super().__init__()
        self.num_input_ts = num_input_ts
        self.num_input_scalar = num_input_scalars
        tcn_layer_cfg = tcn_layer_cfg or {
            "f": [
                {"units": (num_input_scalars + 1), "act_func": "tanh"},
                {"units": 8, "act_func": "tanh"},
                {"units": 1},
            ]
        }
        scalar_layer_cfg = scalar_layer_cfg or {
            "f": [
                {"units": num_input_scalars, "act_func": "tanh"},
            ]
        }
        # build CNN layer path
        cnn_layers = []
        dilation_offset = tcn_layer_cfg.get("starting_dilation_rate", 2)  # >= 0
        for i, l_cfg in enumerate(tcn_layer_cfg["f"]):
            kernel_size = l_cfg.get("kernel_size", 9)
            dropout_rate = tcn_layer_cfg.get("dropout", 0.0)
            dilation_size = 2 ** (i + dilation_offset)
            if i == 0:
                in_channels = num_input_ts
            else:
                in_channels = tcn_layer_cfg["f"][i - 1]["units"]
            cnn_layers += [
                TemporalBlock(
                    in_channels,
                    l_cfg["units"],
                    kernel_size,
                    stride=1,
                    dilation=dilation_size,
                    residual=tcn_layer_cfg.get("residual", False),
                    double_layered=tcn_layer_cfg.get("double_layered", False),
                    dropout=dropout_rate,
                    act_func=l_cfg.get("act_func", nn.Identity),
                ),
            ]
            if i == 0:
                self.ts_branch = cnn_layers.pop()
        self.upper_tcn = nn.Sequential(*cnn_layers)
        # build scalar NN path
        scalar_layers = []
        fan_in = num_input_scalars
        for i, l_cfg in enumerate(scalar_layer_cfg["f"]):
            scalar_layers.append(nn.Linear(fan_in, l_cfg["units"]))
            scalar_layers.append(ACTIVATION_FUNCS.get(l_cfg["act_func"], nn.Identity)())
            fan_in = l_cfg["units"]

        self.scalar_branch = nn.Sequential(*scalar_layers)

    def forward(self, x_ts, x_scalars):
        """x_ts has shape (#batch, #channels, #length)"""
        b_proc = self.ts_branch(x_ts)
        scalar_proc = self.scalar_branch(x_scalars)
        catted = torch.cat(
            [
                b_proc[:, : -scalar_proc.shape[1], :],
                b_proc[:, -scalar_proc.shape[1] :, :] + scalar_proc.unsqueeze(-1),
            ],
            dim=1,
        )
        y = self.upper_tcn(catted)
        y = y + x_ts[:, [0], :]
        y = y - y.mean(dim=-1).unsqueeze(-1)
        return y

class LossPredictor(nn.Module):
    def __init__(
        self,
        h_predictor,
    ):
        super().__init__()
        self.h_predictor = h_predictor
        self.post_processor = nn.Sequential(
            nn.Linear(self.h_predictor.num_input_scalar, 8),
            nn.Tanh(),
            nn.Linear(8, 1),
            nn.Tanh()
        )

    def forward(self, x_ts, x_scalars, b_lim, h_lim, freq_scale):
        h_pred = self.h_predictor(x_ts, x_scalars).permute(2, 0, 1)

        # scalars: Freq (0), Temp (1), Hdc (2)
        # freq = freq_scale * torch.exp(x_scalars[:, [0]]) # Assuming log frequency input?
        # In Paderborn code, they passed log(freq) in x_scalars and also freq_scale.
        # Here we will adapt.

        freq = freq_scale * torch.exp(x_scalars[:, [0]])

        scaled_b = x_ts[:, [-1], :].permute(2, 0, 1)  # globally scaled B curve

        # Paderborn uses arbitrary offsets to make Shoelace positive?
        b_with_offset = b_lim * scaled_b + 5
        h_with_offset = h_lim * h_pred + 5

        ploss_pred = (
            freq
            * (0.5 + 0.1*self.post_processor(x_scalars))
            * torch.abs(
                torch.sum(
                    b_with_offset
                    * (
                        torch.roll(h_with_offset, 1, dims=0)
                        - torch.roll(h_with_offset, -1, dims=0)
                    ),
                    dim=0,
                )
            )
        )
        return torch.log(ploss_pred + 1e-6)

# ==========================================
# Validated Model Wrapper
# ==========================================

class CNNNetwork(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=32, output_dim=1):
        super(CNNNetwork, self).__init__()

        # Paderborn Defaults
        # scalars: Freq, Temp, Hdc = 3

        self.tcn = TCNWithScalarsAsBias(
            num_input_scalars=3,
            num_input_ts=input_dim
        )
        self.model = LossPredictor(self.tcn)

        # Hyperparameters for normalization (Placeholder - should ideally come from dataset stats)
        # We use reasonable defaults from Paderborn
        self.register_buffer('b_lim', torch.tensor(0.5))
        self.register_buffer('h_lim', torch.tensor(150.0))
        self.register_buffer('freq_scale', torch.tensor(150000.0))

    def forward(self, b_seq, scalars):
        # b_seq: (Batch, Seq, 1)
        # scalars: (Batch, 3) -> Freq, Temp, Hdc

        # 1. Adapt B-Sequence -> (Batch, Channels, Length) for TCN
        x_ts = b_seq.permute(0, 2, 1) # (Batch, 1, Seq)

        # 2. Adapt Scalars
        # Paderborn expects Normalized scalars. Dataset returns normalized scalars.
        # But Paderborn expects log(Freq). Dataset provides MinMax Freq?
        # We assume dataset provides appropriate normalized scalars.
        # If dataset provides linear normalized Freq, we might need to adjust.
        # For now, pass as is.

        # 3. Forward
        # LossPredictor returns log_loss
        log_loss = self.model(x_ts, scalars, self.b_lim, self.h_lim, self.freq_scale)

        return log_loss

### Module: `src\models\transformer_model.py`
Content from local file: `src\models\transformer_model.py`

In [12]:

import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """
    Injects some information about the relative or absolute position of the tokens in the sequence.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:x.size(1)]
        return self.dropout(x)

class TransformerNetwork(nn.Module):
    """
    Fuzhou's Transformer-based architecture.
    """
    def __init__(self,
        B_in_channel=1024, # Default sequence length
        dim_hidden=24,
        dim_proj_fusion=40,
        n_encoder_layers=1,
        n_heads=4,
        dropout_encoder=0.0,
        dropout_pos_enc=0.0,
        dim_feedforward_encoder=40,
        ):
        super().__init__()

        # Projection for B-field input: maps scalar input to hidden dimension
        self.proj_B = nn.Sequential(
            nn.Linear(1, dim_hidden),
            nn.Tanh(),
            nn.Linear(dim_hidden, dim_hidden))

        self.positional_encoding_layer = PositionalEncoding(d_model=dim_hidden,
                                                            dropout=dropout_pos_enc,
                                                            max_len=B_in_channel)

        # Transformer Encoder Layer definition
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim_hidden,
            nhead=n_heads,
            dim_feedforward=dim_feedforward_encoder,
            dropout=dropout_encoder,
            activation="relu",
            batch_first=True
            )

        self.encoder = nn.TransformerEncoder(encoder_layer=encoder_layer,
                                             num_layers=n_encoder_layers,
                                             norm=None)

        # Fusion Layer: Combines Transformer output (B-field features) with Temp and Freq
        self.proj_fusion = nn.Sequential(
            nn.Linear(dim_hidden+2, dim_proj_fusion),
            nn.Tanh(),
            nn.Linear(dim_proj_fusion, dim_proj_fusion),
            nn.Tanh(),
            nn.Linear(dim_proj_fusion, 1))

        # Final Regressor to predict Power Loss from fused features
        self.regressor = nn.Sequential(
            nn.Linear(B_in_channel, 1))

    def forward(self, b_seq, scalars):
        """
        Unified Interface Wrapper.

        Args:
            b_seq (Tensor): Input B-field curve, shape (batch_size, seq_len, 1).
            scalars (Tensor): Shape (batch_size, 3) -> Freq, Temp, Hdc.
        """
        # Unpack scalars
        # Dataset returns: Freq, Temp, Hdc
        Freq = scalars[:, 0].unsqueeze(1) # (bs, 1)
        Temp = scalars[:, 1].unsqueeze(1) # (bs, 1)

        B_curve = b_seq

        batch_size, len_seq, feat_dim = B_curve.shape

        # Fuzhou Logic
        B_curve = self.proj_B(B_curve) # (bs,1024,1)->(bs,1024,24)

        # Add Positional Encoding
        B_curve = self.positional_encoding_layer(B_curve)
        B_curve = self.encoder(B_curve)

        # Repeat Temp and Freq to match sequence length for concatenation
        Temp_rep = Temp.unsqueeze(1).repeat(1, len_seq, 1) # (bs,1)->(bs,1024,1)
        Freq_rep = Freq.unsqueeze(1).repeat(1, len_seq, 1) # (bs,1)->(bs,1024,1)

        # Fuse B-field features with Temp and Freq
        feat = self.proj_fusion(torch.cat([B_curve, Temp_rep, Freq_rep], dim=2)) # (bs,1024,26)->(bs,1024,1)
        feat = feat.reshape(batch_size, -1)
        P_pred = self.regressor(feat) # (bs,1)

        return P_pred

### Module: `src\training\train.py`
Content from local file: `src\training\train.py`

In [13]:
"""
Training Loop Module.

This module contains the logic for training the neural networks.
It includes the training loop, validation step, loss calculation, and checkpointing.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os
import copy

def train_model(model, train_loader, val_loader, config, device='cpu'):
    """
    Generic training loop.

    Args:
        model (nn.Module): The model to train.
        train_loader (DataLoader): Training data.
        val_loader (DataLoader): Validation data.
        config (dict): Configuration dictionary (lr, epochs, save_dir).
        device (str): 'cpu' or 'cuda'.

    Returns:
        model (nn.Module): Trained model (best weights).
        history (dict): Training history (loss).
    """
    model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.get('learning_rate', 0.001))

    num_epochs = config.get('epochs', 100)
    save_dir = config.get('save_dir', 'checkpoints')
    os.makedirs(save_dir, exist_ok=True)

    best_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        running_loss = 0.0

        # Use tqdm for progress bar if interactive
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch in pbar:
            if len(batch) == 3:
                inputs, scalars, targets = batch
                inputs = inputs.to(device)
                scalars = scalars.to(device)
                targets = targets.to(device)
                outputs = model(inputs, scalars)
            else:
                inputs, targets = batch
                inputs = inputs.to(device)
                targets = targets.to(device)
                outputs = model(inputs)

            optimizer.zero_grad()
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            pbar.set_postfix({'loss': loss.item()})

        epoch_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_loss)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                if len(batch) == 3:
                    inputs, scalars, targets = batch
                    inputs = inputs.to(device)
                    scalars = scalars.to(device)
                    targets = targets.to(device)
                    outputs = model(inputs, scalars)
                else:
                    inputs, targets = batch
                    inputs = inputs.to(device)
                    targets = targets.to(device)
                    outputs = model(inputs)

                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)

        epoch_val_loss = val_loss / len(val_loader.dataset)
        history['val_loss'].append(epoch_val_loss)

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Val Loss: {epoch_val_loss:.4f}")

        # Deep Copy Best Model
        if epoch_val_loss < best_loss:
            best_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

### Module: `src\training\evaluate.py`
Content from local file: `src\training\evaluate.py`

In [14]:
"""
Evaluation Module.

This module provides functions to evaluate model performance on test datasets.
"""

import torch
import torch.nn as nn
import numpy as np

def evaluate_model(model, test_loader, device='cpu'):
    """
    Evaluates the model and returns predictions and metrics.

    Args:
        model (nn.Module): Trained model.
        test_loader (DataLoader): Test data.
        device (str): Device.

    Returns:
        metrics (dict): MSE, MAE, Relative Error.
        predictions (list): List of pred tensors.
        targets (list): List of target tensors.
    """
    model.eval()
    model.to(device)

    preds = []
    actuals = []

    criterion_mse = nn.MSELoss()
    criterion_mae = nn.L1Loss()

    total_mse = 0.0
    total_mae = 0.0

    with torch.no_grad():
        for batch in test_loader:
            if len(batch) == 3:
                inputs, scalars, targets = batch
                inputs = inputs.to(device)
                scalars = scalars.to(device)
                targets = targets.to(device)
                outputs = model(inputs, scalars)
            else:
                inputs, targets = batch
                inputs = inputs.to(device)
                targets = targets.to(device)
                outputs = model(inputs)

            mse = criterion_mse(outputs, targets)
            mae = criterion_mae(outputs, targets)

            total_mse += mse.item() * inputs.size(0)
            total_mae += mae.item() * inputs.size(0)

            preds.append(outputs.cpu().numpy())
            actuals.append(targets.cpu().numpy())

    num_samples = len(test_loader.dataset)
    avg_mse = total_mse / num_samples
    avg_mae = total_mae / num_samples

    metrics = {
        'mse': avg_mse,
        'mae': avg_mae,
        'rmse': np.sqrt(avg_mse)
    }

    return metrics, np.concatenate(preds), np.concatenate(actuals)

### Module: `src\utils\visualization.py`
Content from local file: `src\utils\visualization.py`

In [15]:
"""
Visualization Module.

This module provides plotting functions for inspecting model performance.
"""

import matplotlib.pyplot as plt
import numpy as np

def plot_loss_curve(history, title='Training History', save_path=None):
    """
    Plots Train vs Val Loss.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(history['train_loss'], label='Train Loss', marker='o')
    plt.plot(history['val_loss'], label='Val Loss', marker='o')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.title(title)
    plt.legend()
    plt.grid(True)

    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_prediction_scatter(preds, targets, title='Predictions vs Actuals', save_path=None):
    """
    Scatter plot for scalar regression.
    """
    plt.figure(figsize=(8, 8))
    plt.scatter(targets, preds, alpha=0.5)

    # Perfect line
    min_val = min(np.min(targets), np.min(preds))
    max_val = max(np.max(targets), np.max(preds))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')

    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title(title)
    plt.grid(True)

    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def plot_bh_loop(pred_b, pred_h, actual_b, actual_h, title='B-H Loop Comparison', save_path=None):
    """
    Plots predicted vs actual B-H loop.
    Args: (Seq_Len,) arrays.
    """
    plt.figure(figsize=(8, 6))
    plt.plot(actual_h, actual_b, 'b-', label='Actual', linewidth=2)
    plt.plot(pred_h, pred_b, 'r--', label='Predicted', linewidth=2)
    plt.xlabel('H (A/m)')
    plt.ylabel('B (T)')
    plt.title(title)
    plt.legend()
    plt.grid(True)

    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

### Module: `main.py`
Content from local file: `main.py`

In [21]:
"""
MagNet Project Entry Point.

This script serves as the main interface for training and evaluating the neural network models.
"""

import argparse
import torch
from torch.utils.data import DataLoader, random_split
# [NOTEBOOK_BUNDLER] imports are handled by the notebook cells defined previously
import yaml
import os

def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def main(args=None):
    parser = argparse.ArgumentParser(description="MagNet Deep Learning Pipeline")
    parser.add_argument('--config', type=str, default='/content/config/config.yaml', help='Path to config file')
    parser.add_argument('--data', type=str, required=True, help='Path to .mat file')
    parser.add_argument('--model', type=str, choices=['scaler', 'sequence', 'seq2seq', 'cnn', 'transformer', 'all'], required=True, help='Model type to train')
    parser.add_argument('--epochs', type=int, help='Override epochs in config')
    args = parser.parse_args(args)

    config = load_config(args.config)

    # Override config
    if args.epochs:
        config['training']['epochs'] = args.epochs

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    models_to_run = [args.model] if args.model != 'all' else ['scaler', 'sequence', 'seq2seq', 'cnn', 'transformer']

    for model_name in models_to_run:
        print(f"\n{'='*20} Training {model_name.upper()} Model {'='*20}")

        # 1. Dataset
        print("Preparing Dataset...")
        dataset = MagNetDataset(args.data, mode=model_name)

        # Split (80/20)
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_set, val_set = random_split(dataset, [train_size, val_size])

        batch_size = config['data'].get('batch_size', 32)
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=batch_size)

        # 2. Model Initialization
        if model_name == 'scaler':
            # Input: Freq, Temp, Hdc (3) -> Output: Log Loss (1)
            model_conf = config['models']['scaler']
            model = ScalerNetwork(input_dim=3, hidden_dim=model_conf['hidden_dim'], num_layers=model_conf['layers'], output_dim=1)

        elif model_name == 'sequence':
            # Input: B (1) -> Output: Log Loss (1)
            model_conf = config['models']['sequence']
            model = SequenceToScalerNetwork(input_dim=1, hidden_dim=model_conf['hidden_dim'], output_dim=1, num_layers=model_conf['num_layers'])

        elif model_name == 'seq2seq':
            # Input: B (1) -> Output: H (1)
            model_conf = config['models']['seq2seq']
            model = Seq2SeqNetwork(input_dim=1, hidden_dim=model_conf['encoder_dim'], output_dim=1)

        # --- FIX STARTS HERE: Unindented 'elif' blocks ---
        elif model_name == 'cnn':
            # Input: B (1) -> Output: Log Loss (1) (Scalar)
            model_conf = config['models']['cnn']
            # Paderborn CNN doesn't use kernel_size/channels from config in the same way,
            # but we can pass them if we update CNNNetwork.
            model = CNNNetwork(input_dim=1)

        elif model_name == 'transformer':
            # Input: B (1) -> Output: Log Loss (1) (Scalar)
            model_conf = config['models']['transformer']
            # Map config keys to Fuzhou Transformer args
            model = TransformerNetwork(
                B_in_channel=1024, # Default seq len
                dim_hidden=model_conf['d_model'],
                n_encoder_layers=model_conf['num_layers'],
                dim_feedforward_encoder=model_conf['dim_feedforward'],
                n_heads=model_conf['nhead'],
                dropout_encoder=model_conf['dropout']
            )
        # --- FIX ENDS HERE ---

        # 3. Train
        print("Starting training...")
        # Subset config for training
        train_config = config['training']
        # We perform a shallow copy or path join to avoid overwriting global config path in loop
        current_save_dir = os.path.join(train_config['save_dir'], model_name)
        if not os.path.exists(current_save_dir):
            os.makedirs(current_save_dir)

        # Update config just for this run (be careful not to mutate original permanently if looping)
        run_config = train_config.copy()
        run_config['save_dir'] = current_save_dir

        model = model.to(device)
        trained_model, history = train_model(model, train_loader, val_loader, run_config, device)

        # 4. Evaluate & Visualize
        print("Evaluating...")
        metrics, preds, targets = evaluate_model(trained_model, val_loader, device)
        print(f"Validation Metrics: {metrics}")

        # Prepare plots directory
        plots_dir = os.path.join(current_save_dir, 'plots')
        os.makedirs(plots_dir, exist_ok=True)

        # Plot Loss
        loss_plot_path = os.path.join(plots_dir, 'loss_curve.png')
        plot_loss_curve(history, title=f'{model_name} Training Loss', save_path=loss_plot_path)
        print(f"Loss plot saved to {loss_plot_path}")

        # Plot Predictions
        if model_name in ['scaler', 'sequence', 'cnn', 'transformer']:
            pred_plot_path = os.path.join(plots_dir, 'prediction_scatter.png')
            plot_prediction_scatter(preds, targets, title=f'{model_name}: Pred vs Actual Loss', save_path=pred_plot_path)
            print(f"Prediction plot saved to {pred_plot_path}")

        elif model_name == 'seq2seq':
            try:
                # Get one batch for visualization
                val_iter = iter(val_loader)
                b_batch, h_batch = next(val_iter)
                b_batch = b_batch.to(device)

                model.eval()
                with torch.no_grad():
                    pred_h_batch = model(b_batch)

                # Take first sample
                sample_idx = 0
                actual_b = b_batch[sample_idx].cpu().squeeze().numpy()
                actual_h = h_batch[sample_idx].cpu().squeeze().numpy() # Target H
                pred_h = pred_h_batch[sample_idx].cpu().squeeze().numpy()

                bh_plot_path = os.path.join(plots_dir, 'bh_loop_comparison.png')
                plot_bh_loop(actual_b, pred_h, actual_b, actual_h, title=f'{model_name} B-H Loop (Norm)', save_path=bh_plot_path)
                print(f"B-H Loop plot saved to {bh_plot_path}")

            except Exception as e:
                print(f"Could not plot B-H loop: {e}")

# if __name__ == "__main__":
#    main()

### Run Training
Call the `main()` function with arguments as a list of strings.

In [ ]:
# Example: Train CNN Model
# Make sure the data file path is correct
data_file = "/content/drive/MyDrive/Colab Notebooks/3C90_TX-25-15-10_Data1_Cycle.mat"
if not os.path.exists(data_file):
    print(f"Warning: {data_file} not found. Please upload it or fix the path.")

# Arguments: --data <path> --model <model_name> --epochs <N>
args = ['--data', data_file, '--model', 'cnn', '--epochs', '10']

try:
    main(args)
except SystemExit:
    # argparse raises SystemExit on help or error, catch it so notebook doesn't crash
    pass

Using device: cpu

==================== Training CNN Model ====================
Preparing Dataset...
Loading dataset from /content/drive/MyDrive/Colab Notebooks/3C90_TX-25-15-10_Data1_Cycle.mat for partition 'train'...
Reading full arrays from HDF5... this may take a moment.
Running centralized preprocessing...
Loaded config for 'cnn': {'B': 'minmax', 'Frequency': 'standard', 'Temperature': 'standard', 'Hdc': 'standard', 'Loss': 'standard'}
--- 1. Physics Calculations ---
Selected Features: ['B', 'Frequency', 'Temperature', 'Hdc']
Selected Targets: ['Loss']
--- 2. Generic Normalization & Split ---
Splitting Dataset: 109411 Samples -> 87528 Train, 21883 Test
Processing Features...
  - Normalizing 'B' with minmax...
  - Normalizing 'Frequency' with standard...
  - Normalizing 'Temperature' with standard...
  - Normalizing 'Hdc' with standard...
Processing Targets...
  - Normalizing Target 'Loss' with standard...
Starting training...


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Epoch 1/10:   0%|          | 0/2189 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch 1/10: 100%|█████████▉| 2188/2189 [01:10<00:00, 37.51it/s, loss=3.59]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_lo

Epoch 1/10 - Train Loss: 14.1666 - Val Loss: 4.2146


Epoch 2/10: 100%|██████████| 2189/2189 [01:10<00:00, 31.09it/s, loss=2.87]


Epoch 2/10 - Train Loss: 6.2076 - Val Loss: 4.1285


...

Epoch 6/10: 100%|██████████| 2189/2189 [01:10<00:00, 31.18it/s, loss=1.75]


Epoch 6/10 - Train Loss: 4.6854 - Val Loss: 5.4023


Epoch 7/10: 100%|██████████| 2189/2189 [01:09<00:00, 31.49it/s, loss=2.49]


Epoch 7/10 - Train Loss: 4.6867 - Val Loss: 5.1539


Epoch 8/10:  95%|█████████▍| 2071/2189 [01:07<00:03, 31.86it/s, loss=4.01]